# v14 PLM — Stage 2 training on vast.ai (run as-is, top to bottom)

End result: `plm_weights.pt` ready to download.

This notebook uses a dedicated **venv** (`/workspace/venv`) and calls its
python EXPLICITLY in every cell — this sidesteps the classic Jupyter trap
where `!pip` installs into a different interpreter than the kernel
imports from (which is exactly what bit us on this image).
The venv is created with `--system-site-packages` so it inherits the
image's CUDA torch instead of re-downloading 3GB.

**Before running — on your Mac:** commit & push everything the run needs:
```
git add CommunitySolutions/chronos_solver/v14 CommunitySolutions/chronos_solver/v13/*.json
git commit -m 'v14 PLM' && git push
```

Private repo? Put a token in `GIT_TOKEN` (Cell 1). All cells use absolute
paths and re-define their own variables — safe to re-run in any order
after a kernel restart. Training runs via `nohup` (survives tab drops).

In [ ]:
# ---------------- Cell 1: clone / update the repo ----------------
import os
GIT_TOKEN = ""          # only needed if the repo is private
BRANCH    = "main"      # branch that contains the v14 folder

url = (f"https://{GIT_TOKEN}@github.com/shreyasmahimkar/arc-agi-3"
       if GIT_TOKEN else "https://github.com/shreyasmahimkar/arc-agi-3")
if not os.path.exists("/workspace/arc3"):
    !git clone --depth 1 --branch {BRANCH} {url} /workspace/arc3
else:
    !cd /workspace/arc3 && git pull
!ls /workspace/arc3/CommunitySolutions/chronos_solver/v14

In [ ]:
# ---------------- Cell 2: venv + dependencies ----------------
# --system-site-packages: inherit the image's CUDA torch (no 3GB re-pull).
# Everything from here on runs through PY — one interpreter, no ambiguity.
import os, sys
PY = "/workspace/venv/bin/python"
if not os.path.exists(PY):
    !{sys.executable} -m venv --system-site-packages /workspace/venv

W = "/workspace/arc3/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"
!{PY} -m pip -q install --ignore-installed blinker
!{PY} -m pip -q install {W}/arcengine-0.9.3-py3-none-any.whl \
    {W}/arc_agi-0.9.8-py3-none-any.whl python-dotenv

# hard verification THROUGH the venv python — fails loudly if anything broke
!{PY} -c "import arcengine, arc_agi, torch; print('deps verified | torch', torch.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
# ---------------- Cell 3: sanity — GPU + module smoke test ----------------
PY = "/workspace/venv/bin/python"
!{PY} -c "import torch; p=torch.cuda.get_device_properties(0); print(torch.cuda.get_device_name(0), f'{p.total_memory/1e9:.1f} GB')"
!cd /workspace/arc3/CommunitySolutions/chronos_solver/v14 && {PY} -m plm.smoke

In [ ]:
# ---------------- Cell 4: generate training data (~5 min) ----------------
PY = "/workspace/venv/bin/python"
!cd /workspace/arc3/CommunitySolutions/chronos_solver/v14 && \
    {PY} gen_data.py --out /workspace/v14_shards \
        --episodes-per-game 400 --max-steps 150
!ls -lh /workspace/v14_shards | tail -5
!df -h /workspace | tail -1

In [ ]:
# ---------------- Cell 5: LAUNCH training (returns immediately) ----------------
# nohup-detached: survives tab drops and kernel restarts.
# Gates: tokenizer pixel_acc >= 0.995, then HELDOUT_tok_acc >= 0.90.
PY = "/workspace/venv/bin/python"
!cd /workspace/arc3/CommunitySolutions/chronos_solver/v14 && \
    nohup {PY} train_wm.py --phase all --shards /workspace/v14_shards \
        --epochs 20 --steps-per-epoch 1000 --bsz 256 \
        --holdout ls20,vc33,tu93,ft09,sp80 > /workspace/train.log 2>&1 &
import time; time.sleep(5)
!tail -3 /workspace/train.log

In [ ]:
# ---------------- Cell 6: MONITOR (re-run me anytime) ----------------
!ps aux | grep train_wm | grep -v grep || echo '*** TRAINING NOT RUNNING (finished or crashed - check log below) ***'
print('-' * 70)
!tail -15 /workspace/train.log
print('-' * 70)
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

In [ ]:
# ---------------- Cell 7: VERIFY + stage for download ----------------
# (run when Cell 6 shows training finished)
PY = "/workspace/venv/bin/python"
p = "/workspace/arc3/CommunitySolutions/chronos_solver/v14/plm_weights.pt"
!{PY} -c "import torch,os; s=torch.load('{p}', map_location='cpu', weights_only=True); print('keys:', list(s)); print(f'size: {{os.path.getsize(\"{p}\")/1e6:.1f}} MB')"
!cp {p} /workspace/plm_weights.pt && cp /workspace/train.log /workspace/train_final.log
print("Download via the Jupyter file browser (/workspace): plm_weights.pt + train_final.log")
print("Then DESTROY this instance on the vast.ai console.")

## After downloading

1. Put `plm_weights.pt` into your local `CommunitySolutions/chronos_solver/v14/`
   (`*.pt` is gitignored — it travels by hand, never via git).
2. Local gate: run the held-out-game eval vs the v13 bandit baseline.
3. Stage 3: update the `v14-plm` Kaggle dataset and submit
   (see DEPLOYMENT.md).